# Validación del Modelo Estrella Kimball — Proyecto 1 DAP

Este notebook valida, de forma reproducible, que:
1. El DDL crea el esquema correctamente.
2. El ETL carga el CSV sin errores y sin filas huérfanas.
3. Las consultas del DML corren y devuelven resultados coherentes.

**Requisitos:** dejar en la misma carpeta que este notebook los archivos
`01_DDL_kimball.sql`, `02_etl_to_sql.py`, `03_DML_analitico.sql` y el CSV
`DS_DAP_P1_clean.csv`.


In [ ]:
import sqlite3
import subprocess
import sys
import pandas as pd
from pathlib import Path

CARPETA = Path(".")
CSV_FILE = CARPETA / "DS_DAP_P1_clean.csv"
DDL_FILE = CARPETA / "01_DDL_kimball.sql"
ETL_FILE = CARPETA / "02_etl_to_sql.py"
DML_FILE = CARPETA / "03_DML_analitico.sql"
DB_FILE  = CARPETA / "proyecto1_kimball.db"

for f in [CSV_FILE, DDL_FILE, ETL_FILE, DML_FILE]:
    print(f"{'OK ' if f.exists() else 'FALTA'}  {f}")


## 1. Ejecutar el ETL y capturar su log

In [ ]:
# Se ejecuta como proceso aparte para que el notebook valide exactamente
# lo que correría un evaluador desde la terminal: python 02_etl_to_sql.py
if DB_FILE.exists():
    DB_FILE.unlink()  # empezar desde cero para que la validación sea limpia

resultado = subprocess.run(
    [sys.executable, str(ETL_FILE)],
    cwd=CARPETA, capture_output=True, text=True
)

print(resultado.stdout)
if resultado.returncode != 0:
    print("=== ERROR ===")
    print(resultado.stderr)

assert resultado.returncode == 0, "El ETL terminó con error, revisar salida arriba"
print("\nETL ejecutado sin errores.")


## 2. Validar el esquema (DDL)
Comprobamos que existan las 4 tablas y que las llaves foráneas queden
correctamente declaradas.

In [ ]:
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON;")

tablas_esperadas = {"Dim_Calendar", "Dim_Agent", "Dim_IssueCategory", "Fact_ServiceEvents"}
tablas_reales = {r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table';")}

faltantes = tablas_esperadas - tablas_reales
assert not faltantes, f"Faltan tablas: {faltantes}"
print("Tablas presentes:", sorted(tablas_reales))

print("\n--- Foreign keys declaradas en Fact_ServiceEvents ---")
for row in conn.execute("PRAGMA foreign_key_list(Fact_ServiceEvents);"):
    print(row)


In [ ]:
# pragma foreign_key_check detecta filas que violan alguna FK,
# incluso si las FKs se cargaron con PRAGMA foreign_keys apagado
violaciones = conn.execute("PRAGMA foreign_key_check;").fetchall()
assert len(violaciones) == 0, f"Hay violaciones de integridad referencial: {violaciones}"
print("Integridad referencial: OK, 0 violaciones (PRAGMA foreign_key_check).")


## 3. Validar el volumen de datos cargado

In [ ]:
conteos = {}
for t in tablas_esperadas:
    conteos[t] = conn.execute(f"SELECT COUNT(*) FROM {t};").fetchone()[0]

df_conteos = pd.DataFrame(conteos.items(), columns=["tabla", "filas"])
display(df_conteos)

# El CSV fuente tiene 2000 tickets: la tabla de hechos debe tener exactamente
# ese número de filas (grano 1 fila = 1 ticket, sin duplicar ni perder)
df_csv = pd.read_csv(CSV_FILE)
assert conteos["Fact_ServiceEvents"] == len(df_csv), (
    f"Fact_ServiceEvents tiene {conteos['Fact_ServiceEvents']} filas, "
    f"pero el CSV origen tiene {len(df_csv)}"
)
print(f"Fact_ServiceEvents coincide con el CSV origen: {len(df_csv)} tickets.")


## 4. Validar que no haya FKs nulas en los hechos
Un `calendar_sk`, `agent_sk` o `issue_sk` en NULL indica que el merge del ETL
no encontró la dimensión correspondiente (mapeo de llaves roto).

In [ ]:
nulos_fk = conn.execute("""
    SELECT
        SUM(CASE WHEN calendar_sk IS NULL THEN 1 ELSE 0 END) AS calendar_sk_nulos,
        SUM(CASE WHEN agent_sk    IS NULL THEN 1 ELSE 0 END) AS agent_sk_nulos,
        SUM(CASE WHEN issue_sk    IS NULL THEN 1 ELSE 0 END) AS issue_sk_nulos
    FROM Fact_ServiceEvents;
""").fetchone()

print("FKs nulas en Fact_ServiceEvents:", nulos_fk)
assert all(n == 0 for n in nulos_fk), "Hay FKs nulas: revisar el merge del ETL"
print("Sin FKs nulas.")


## 5. Validar reglas de negocio en las métricas
Chequeos de sanidad: rangos de csat, banderas 0/1, y que
`total_cycle_time_hrs ≈ wait_time_hrs + handling_time_hrs` cuando el ticket
está cerrado.

In [ ]:
fact = pd.read_sql("SELECT * FROM Fact_ServiceEvents", conn)

# csat_score entre 1 y 5 (o NULL)
csat_fuera_rango = fact[fact["csat_score"].notna() &
                        ~fact["csat_score"].between(1, 5)]
assert csat_fuera_rango.empty, f"{len(csat_fuera_rango)} filas con csat_score fuera de 1-5"

# sla_compliant y fcr_flag deben ser 0, 1 o NULL (sla_compliant) / 0,1 (fcr_flag)
assert fact["sla_compliant"].dropna().isin([0, 1]).all(), "sla_compliant con valores inválidos"
assert fact["fcr_flag"].isin([0, 1]).all(), "fcr_flag con valores inválidos"

# Coherencia de tiempos en tickets cerrados (tolerancia por redondeo)
cerrados = fact.dropna(subset=["wait_time_hrs", "handling_time_hrs", "total_cycle_time_hrs"])
diff = (cerrados["wait_time_hrs"] + cerrados["handling_time_hrs"]
        - cerrados["total_cycle_time_hrs"]).abs()
inconsistentes = cerrados[diff > 0.5]  # más de 0.5 hrs de diferencia
print(f"Tickets cerrados evaluados: {len(cerrados)}")
print(f"Inconsistentes (wait+handling != total, tolerancia 0.5h): {len(inconsistentes)}")

print("\nValidaciones de negocio: OK")


## 6. Ejecutar el DML completo y mostrar cada resultado

In [ ]:
def dividir_sql(texto):
    """Separa el script en sentencias individuales, ignorando bloques
    que sólo contienen comentarios (los separadores de sección del archivo)."""
    partes = [p.strip() for p in texto.split(';')]
    consultas = []
    for p in partes:
        lineas = [l for l in p.splitlines() if l.strip() and not l.strip().startswith('--')]
        if lineas:
            consultas.append(p)
    return consultas

sql_dml = DML_FILE.read_text(encoding="utf-8")
consultas = dividir_sql(sql_dml)
print(f"{len(consultas)} consultas detectadas en {DML_FILE.name}\n")

resultados = {}
for idx, q in enumerate(consultas):
    try:
        df_r = pd.read_sql(q, conn)
        resultados[idx] = df_r
        print(f"Consulta {idx}: OK -> {df_r.shape[0]} filas, {df_r.shape[1]} columnas")
    except Exception as e:
        print(f"Consulta {idx}: FALLA -> {e}")


### 6.1 KPIs globales (Vista Ejecutiva)

In [ ]:
display(resultados[0])

### 6.2 Pareto 80/20 por categoría

In [ ]:
df_pareto = resultados[1]
display(df_pareto)

vitales = df_pareto[df_pareto["clase_pareto"] == "POCOS VITALES (20%)"]
print(f"\n{len(vitales)} de {len(df_pareto)} categorías concentran el 80% de las horas de retraso:")
print(", ".join(vitales["category"].tolist()))


### 6.3 Rendimiento por agente

In [ ]:
display(resultados[2])

### 6.4 Cuellos de botella por canal y prioridad

In [ ]:
display(resultados[3])

### 6.5 Heatmap día de la semana x hora (insumo del dashboard)

In [ ]:
df_heatmap = resultados[4]
tabla_heatmap = df_heatmap.pivot(index="day_of_week", columns="created_hour", values="volumen").fillna(0)
orden_dias = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]
tabla_heatmap = tabla_heatmap.reindex(orden_dias)
display(tabla_heatmap.style.background_gradient(cmap="Oranges", axis=None))


### 6.6 Serie diaria con promedio móvil de 7 días

In [ ]:
df_ma7 = resultados[5]
df_ma7["date_key"] = pd.to_datetime(df_ma7["date_key"])

ax = df_ma7.set_index("date_key")[["tickets", "ma7_tickets"]].plot(
    figsize=(11, 4), title="Volumen diario de tickets vs. promedio móvil 7 días")
ax.set_xlabel("Fecha"); ax.set_ylabel("Tickets")


### 6.7 Auditoría de calidad de datos

In [ ]:
display(resultados[6])

## 7. Resumen final de la validación

In [ ]:
print("=" * 60)
print("RESUMEN DE VALIDACIÓN - MODELO ESTRELLA KIMBALL")
print("=" * 60)
print(f"Tablas creadas         : {len(tablas_reales)} / 4 esperadas")
print(f"Filas Fact_ServiceEvents: {conteos['Fact_ServiceEvents']} (= filas del CSV origen)")
print(f"Violaciones de FK       : {len(violaciones)} (esperado: 0)")
print(f"FKs nulas en hechos     : {sum(nulos_fk)} (esperado: 0)")
print(f"Consultas DML ejecutadas: {len(resultados)} / {len(consultas)}")
print("=" * 60)
print("Validación completa: el modelo estrella cumple integridad y las")
print("consultas analíticas corren correctamente sobre proyecto1_kimball.db")

conn.close()
